# M4: Estudio del parámetro de penalización λ

Según el plan (M4):
- Umbral exacto $λ^* = \max_{x\,infeasible}\frac{C_{opt} - C(x)}{P(x)}$ con $P(x)=\sum_i (\sum_j x_{ij}-1)^2 + \sum_j (\sum_i x_{ij}-1)^2$. Para $λ > λ^*$ el estado fundamental del QUBO es factible y óptimo.
- Elegir $λ = 1.5$ - $2\timesλ^*$; para N=10 se esperan decenas.
- Gráficas en `docs/figures/lambda_*.png`:
  1) $λ^*$ vs N
  2) Para QAOA con N=3/4: fracción factible y mejor costo factible vs $λ \in \{0.5, 1, 2, 4\}\timesλ^*$.

**Mensaje:** demasiado pequeño → óptimo infactible. Demasiado grande → paisaje plano, QAOA tiene dificultades. Hay que justificar el valor elegido.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import matplotlib.pyplot as plt

from qap.model import build_qp, to_qubo, penalty_threshold
from qap.types import Instance
from qap.quantum import solve_qaoa

F = np.array([[0,10,4,8],[0,0,6,0],[0,0,0,0],[0,0,0,0]])
D = np.array([[0,1,2,3],[1,0,1,2],[2,1,0,1],[3,2,1,0]])

def instance(n):
    return Instance(f'test{n}', n, F[:n,:n].copy(), D[:n,:n].copy())

## 1) $λ^*$ vs N (N=3,4)

In [ ]:
lambdas_star = {}
for n in [3,4]:
    lambdas_star[n] = penalty_threshold(instance(n))
    print(f'N={n}: lambda* = {lambdas_star[n]:.3f}')

plt.figure(figsize=(4,3))
plt.plot(list(lambdas_star.keys()), list(lambdas_star.values()), marker='o')
plt.xlabel('N')
plt.ylabel('lambda*')
plt.title('Umbral exacto de penalización')
plt.grid(True, alpha=0.3)
plt.savefig('../docs/figures/lambda_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

## 2) QAOA con N=3 y N=4 vs $λ / λ^*$

Usamos el **Camino B** (`solve_qaoa`). Para cada caso medimos la fracción factible de los shots y el mejor costo factible. Semillas fijas para reproducibilidad.

In [ ]:
factors = [0.5, 1.0, 2.0, 4.0]
results = []

for n in [3,4]:
    inst = instance(n)
    ls = lambdas_star[n]
    for f in factors:
        lam = ls * f
        print(f'Running N={n} lambda/lambda*={f} (lambda={lam:.3f})...')
        res = solve_qaoa(inst, seed=0, reps=1, maxiter=100, shots=4096, penalty=lam)
        results.append({
            'n': n,
            'factor': f,
            'lambda': lam,
            'feasible_fraction': res.extras.get('feasible_fraction', 0),
            'best_feasible_cost': res.cost,
            'optimal_prob': res.extras.get('optimal_prob', 0),
            'seconds': res.seconds
        })
        print(results[-1])

## Gráficas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,6))

for i,n in enumerate([3,4]):
    sub = [r for r in results if r['n']==n]
    factors_n = [r['factor'] for r in sub]
    ff = [r['feasible_fraction'] for r in sub]
    bfc = [r['best_feasible_cost'] for r in sub]
    opt = instance(n).optimum
    
    ax = axes[0,i]
    ax.plot(factors_n, ff, marker='o')
    ax.set_xlabel('lambda / lambda*')
    ax.set_ylabel('fracción factible')
    ax.set_title(f'N={n} - fracción factible')
    ax.grid(True, alpha=0.3)
    
    ax2 = axes[1,i]
    ax2.plot(factors_n, bfc, marker='o', label='mejor costo factible')
    ax2.axhline(opt, ls='--', color='r', label=f'óptimo {opt}')
    ax2.set_xlabel('lambda / lambda*')
    ax2.set_ylabel('costo')
    ax2.set_title(f'N={n} - mejor costo factible')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/figures/lambda_qaoa_feasibility.png', dpi=150, bbox_inches='tight')
plt.show()

## Recomendación

Basándonos en los resultados:
- Con $λ << λ^*$ (factor 0.5) las restricciones no se penalizan lo suficiente → el óptimo del paisaje puede ser infactible.
- Con $λ > λ^*$ (factor >= 2) el espacio factible se prioriza correctamente. Sin embargo, penalizaciones muy grandes aplanan el paisaje y dificultan la optimización de QAOA.
- **Recomendado:** $λ = 1.5$ - $2\timesλ^*$.

Para N=10 se extrapola el orden de magnitud: $λ^*$ crecerá, por lo que se esperan valores en las decenas.